# IEC 61850 packet-batch Random Forest

This notebook decodes the supplied PCAPNG files directly, uses **every PCAPNG in `Training Dataset`** to fit separate GOOSE and SV Random Forests, then evaluates on `Test Dataset/test.pcapng`. SV frames may carry many ASDUs; they are decoded completely and summarized into packet and packet-batch features. No event CSV is required.

The SV model uses causal counter, timing, publisher-change, and waveform-continuity features. Six configured float32 channels are read from each ASDU’s `seqData` at byte offsets 0, 4, 8, 12, 16, and 20. Features emphasize changes from the same ASDU position in earlier frames (relative jumps, plateaus, robust change ratios, and within-frame channel ranges), rather than treating raw magnitudes as device identity. These offsets are verified for the supplied lab captures; units are not known and must be checked before site use.

Labels are the supplied lab scenario labels. GOOSE `test` and the SV S-bit are used only to assign these provisional labels and are excluded from model features. Sender-controlled markers are weak evidence, so scores are research results for these captures, not operational detection claims.

Run all cells in order. Edit the single `CONFIG` cell below for capture paths, labels, channel layout, packet batch size, feature thresholds, Random Forest settings, and outputs.

## 1. Editable configuration

In [ ]:
from pathlib import Path
import copy
import json
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "tools" / "pcap_events.py").is_file():
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools" / "pcap_events.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT to the repository containing tools/pcap_events.py")
TOOLS_DIR = REPO_ROOT / "tools"
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from lab_capture_labels import LAB_LABEL_CONFIG

CONFIG = {
    "paths": {
        "training_dir": REPO_ROOT / "Training Dataset",
        "test_pcap": REPO_ROOT / "Test Dataset" / "test.pcapng",
        "output_dir": REPO_ROOT / "rf_batch_models",
        "expected_training_captures": [
            "baseline_new.pcapng", "goosedelta500.pcapng",
            "gooseSVcorrmismatch.pcapng", "pureSV.pcapng",
            "svspoof.pcapng", "traininglegitfault.pcapng", "tripspoof.pcapng",
        ],
    },
    "labels": copy.deepcopy(LAB_LABEL_CONFIG),
    # Rolling 4-frame context, refreshed every 2 frames. The first alert needs
    # four stream frames; end-of-stream partial tails are not emitted.
    "batch": {"size_frames": 4, "stride_frames": 2, "emit_partial": False},
    "sv_channels": {
        # Verified float32 offsets for supplied lab captures. Engineering units
        # and this layout must be independently checked for a new installation.
        "offsets": [0, 4, 8, 12, 16, 20],
        "format": ">f",
        "relative_scale": [1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
        "units": "as encoded; engineering units not independently verified",
    },
    "waveform": {
        "epsilon": 1e-6, "relative_jump_limit": 0.05,
        "plateau_relative_limit": 1e-6, "history_min_samples": 16,
        "history_samples": 128, "robust_ratio_cap": 100.0,
    },
    "counter": {"max_smp_delta": 100, "max_stnum_step": 20},
    "timing": {"missing_age_s": 10.0, "history_frames": 32},
    "threshold": {"method": "capture_grouped_oof_max_fpr", "max_validation_fpr": 0.01},
    "model": {
        "n_estimators": 100, "max_depth": 12, "min_samples_leaf": 3,
        "max_features": "sqrt", "class_weight": "balanced_subsample",
        "random_state": 42, "n_jobs": -1,
    },
}
print("Repository:", REPO_ROOT)
print(json.dumps({k: v for k, v in CONFIG.items() if k != "labels"}, indent=2, default=str))
print("Editable capture label rules:", json.dumps(CONFIG["labels"], indent=2))

## 2. Decode and label every supplied capture

In [ ]:
from collections import Counter
import time
from pcap_events import decode_many
from lab_capture_labels import label_events

train_dir = Path(CONFIG["paths"]["training_dir"])
test_path = Path(CONFIG["paths"]["test_pcap"])
train_paths = sorted(train_dir.glob("*.pcapng"))
expected_names = set(CONFIG["paths"]["expected_training_captures"])
actual_names = {p.name for p in train_paths}
if not train_dir.is_dir() or actual_names != expected_names:
    raise FileNotFoundError(f"Training PCAP set mismatch; expected={sorted(expected_names)}, found={sorted(actual_names)}")
if not test_path.is_file():
    raise FileNotFoundError(f"Required test PCAPNG is missing: {test_path}")

decode_started = time.perf_counter()
train_raw, train_diagnostics = decode_many(train_paths)
test_raw, test_diagnostics = decode_many([test_path])
decode_elapsed_s = time.perf_counter() - decode_started
print(f"Decoded {len(train_raw) + len(test_raw):,} events in {decode_elapsed_s:.2f}s ({(len(train_raw) + len(test_raw)) / max(decode_elapsed_s, 1e-9):,.0f} events/s)")
for report in train_diagnostics + test_diagnostics:
    print(json.dumps(report, indent=2))
    if report["malformed_iec_frames"]:
        raise ValueError(f"Malformed IEC frame(s) in {report['source_pcap']}; refusing silent partial training")
    if report["unsupported_linktype"]:
        raise ValueError(f"Unsupported link type in {report['source_pcap']}; refusing partial training")
if any(not __import__("math").isfinite(float(row["timestamp"])) for row in train_raw + test_raw):
    raise ValueError("A required packet has no finite capture timestamp")

train_events = label_events(train_raw, "train", config=CONFIG["labels"],
    expected_training_captures=CONFIG["paths"]["expected_training_captures"], test_capture=test_path.name)
test_events = label_events(test_raw, "test", config=CONFIG["labels"],
    expected_training_captures=CONFIG["paths"]["expected_training_captures"], test_capture=test_path.name)
if {r["source_pcap"] for r in train_events} != expected_names:
    raise ValueError("Not every expected training PCAP contributed decoded events")
if any(str(r.get("attack", "")) not in {"0", "1"} for r in train_events):
    raise ValueError("Training labels must be concrete 0/1 labels")
print("Training decoded event counts:", Counter(r["protocol"] for r in train_events))
print("Test decoded event counts:", Counter(r["protocol"] for r in test_events))
print("Test unlabeled/ambiguous events:", sum(str(r.get("attack", "")) not in {"0", "1"} for r in test_events))

# Audit marker counts by unique Ethernet frame; markers are labels only.
for label, rows in (("train", train_events), ("test", test_events)):
    marker_frames = {}
    for row in rows:
        marker = row.get("raw_test") if row["protocol"] == "GOOSE" else row.get("raw_simulated")
        if str(marker).strip().lower() in {"true", "1", "yes"}:
            marker_frames[(row["source_pcap"], row["protocol"], row["frame_index"])] = 1
    print(label, "marker-positive frames (audit only):", Counter(k[1] for k in marker_frames))

## 3. Build causal packet batches

In [ ]:
from batch_pipeline import (
    StatefulBatchFeatureBuilder, build_batches, feature_names,
    GOOSE_FRAME_FEATURES, SV_FRAME_FEATURES,
)

# Preserve the wire order from each capture and never mix capture state.
feature_started = time.perf_counter()
train_batches = build_batches(train_events, CONFIG)
test_batches = build_batches(test_events, CONFIG)
feature_elapsed_s = time.perf_counter() - feature_started
print(f"Built {len(train_batches) + len(test_batches):,} packet batches in {feature_elapsed_s:.2f}s ({(len(train_batches) + len(test_batches)) / max(feature_elapsed_s, 1e-9):,.0f} batches/s)")
print("Training batches by protocol:", Counter(r["protocol"] for r in train_batches))
print("Test batches by protocol:", Counter(r["protocol"] for r in test_batches))
print("Training capture coverage:", sorted({r["capture_id"] for r in train_batches}))
if {r["capture_id"] for r in train_batches} != expected_names:
    raise ValueError("At least one training PCAP produced no batch")

# Labels are any-attack over all packets/ASDUs in the batch. Test metrics omit
# batches containing an unknown label, while predictions are still generated.
def feature_matrix(rows, protocol, names=None):
    selected = [r for r in rows if r["protocol"] == protocol]
    if names is None:
        names = feature_names(protocol)
    X = [[float(r[name]) for name in names] for r in selected]
    return selected, names, X

for split, rows in (("train", train_batches), ("test", test_batches)):
    print(split, "batch labels:", {p: Counter(str(r["attack"]) for r in rows if r["protocol"] == p) for p in ("GOOSE", "SV")})

## 4. Fit per-protocol Random Forests on all training PCAPs

In [ ]:
import hashlib
import joblib
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import LeaveOneGroupOut

def capture_balanced_weights(rows):
    counts = Counter(r["capture_id"] for r in rows)
    return np.asarray([1.0 / counts[r["capture_id"]] for r in rows], dtype=float)

def make_forest():
    m = CONFIG["model"]
    return RandomForestClassifier(
        n_estimators=m["n_estimators"], max_depth=m["max_depth"],
        min_samples_leaf=m["min_samples_leaf"], max_features=m["max_features"],
        class_weight=m["class_weight"], random_state=m["random_state"], n_jobs=m["n_jobs"],
    )

def score_metrics(labels, predictions):
    labels, predictions = np.asarray(labels, dtype=int), np.asarray(predictions, dtype=int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    return {"support": int(len(labels)), "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
        "false_positive_rate": float(fp / (fp + tn)) if fp + tn else None}

def select_oof_threshold(y, probabilities, groups, max_fpr):
    # Choose a conservative threshold that meets the target in each training
    # capture's held-out OOF predictions, not merely in the pooled rows.
    y = np.asarray(y, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    groups = np.asarray(groups)
    thresholds = {}
    for capture in sorted(set(groups)):
        negatives = np.sort(probabilities[(groups == capture) & (y == 0)])[::-1]
        if not len(negatives):
            raise ValueError(f"OOF threshold selection needs benign rows in capture {capture}")
        allowed_fp = int(np.floor(max_fpr * len(negatives)))
        thresholds[capture] = 0.0 if allowed_fp >= len(negatives) else float(np.nextafter(negatives[allowed_fp], np.inf))
    return max(thresholds.values()), thresholds

def grouped_oof_threshold(rows, X, y, max_fpr):
    groups = np.asarray([r["capture_id"] for r in rows])
    oof = np.full(len(rows), np.nan, dtype=float)
    for train_idx, valid_idx in LeaveOneGroupOut().split(X, y, groups):
        if len(np.unique(y[train_idx])) != 2:
            raise ValueError("A leave-one-capture-out training fold lacks one class")
        fold = make_forest()
        fold.fit(X[train_idx], y[train_idx], sample_weight=capture_balanced_weights([rows[i] for i in train_idx]))
        oof[valid_idx] = fold.predict_proba(X[valid_idx])[:, list(fold.classes_).index(1)]
    if not np.isfinite(oof).all():
        raise RuntimeError("Incomplete capture-grouped OOF probabilities")
    threshold, capture_thresholds = select_oof_threshold(y, oof, groups, max_fpr)
    oof_prediction = (oof >= threshold).astype(int)
    oof_global = score_metrics(y, oof_prediction)
    capture_fprs = {}
    for capture in sorted(set(groups)):
        mask = groups == capture
        capture_fprs[capture] = score_metrics(y[mask], oof_prediction[mask])["false_positive_rate"]
        if capture_fprs[capture] is not None and capture_fprs[capture] > max_fpr + 1e-12:
            raise RuntimeError(f"Selected OOF threshold violates configured maximum FPR in {capture}")
    fold_metrics = []
    for capture in sorted(set(groups)):
        mask = groups == capture
        fold_metrics.append({"capture_id": capture, "support": int(mask.sum()),
            "metrics_at_oof_threshold": score_metrics(y[mask], (oof[mask] >= threshold).astype(int))})
    return threshold, oof, fold_metrics, oof_global, capture_thresholds, capture_fprs

def feature_matrix(rows, protocol, names=None):
    selected = [r for r in rows if r["protocol"] == protocol]
    if names is None:
        names = feature_names(protocol)
    X = np.asarray([[float(r[name]) for name in names] for r in selected], dtype=float)
    if not np.isfinite(X).all():
        raise ValueError(f"Nonfinite model feature in {protocol} batches")
    return selected, tuple(names), X

models = {}
evaluation = {"label_quality": "provisional lab scenario labels; sender-controlled markers",
              "threshold_method": CONFIG["threshold"], "protocols": {}}
prediction_rows = []
for protocol in ("GOOSE", "SV"):
    tr, names, X_train = feature_matrix(train_batches, protocol)
    te, _, X_test = feature_matrix(test_batches, protocol, names)
    y_train = np.asarray([int(r["attack"]) for r in tr], dtype=int)
    if len(set(y_train)) != 2:
        raise ValueError(f"{protocol} training batches must contain both classes")
    threshold, oof, folds, oof_global, capture_thresholds, capture_fprs = grouped_oof_threshold(
        tr, X_train, y_train, CONFIG["threshold"]["max_validation_fpr"])
    model = make_forest()
    model.fit(X_train, y_train, sample_weight=capture_balanced_weights(tr))
    probabilities = model.predict_proba(X_test)[:, list(model.classes_).index(1)]
    predictions = (probabilities >= threshold).astype(int)
    known = np.asarray([str(r.get("attack", "")) in {"0", "1"} for r in te])
    y_test = np.asarray([int(r["attack"]) if str(r.get("attack", "")) in {"0", "1"} else -1 for r in te])
    test_metric = score_metrics(y_test[known], predictions[known]) if known.any() else {"support": 0}
    details = {
        "threshold": threshold, "threshold_provenance": "leave-one-capture-out OOF predictions on Training Dataset only",
        "train_batches": len(tr), "train_captures": sorted({r["capture_id"] for r in tr}),
        "train_label_counts": dict(Counter(map(int, y_train))), "capture_grouped_oof_folds": folds,
        "capture_grouped_oof_metrics": oof_global,
        "test_batches": len(te), "test_known_label_batches": int(known.sum()),
        "test_unknown_batches": int((~known).sum()), "test_metrics_known_labels": test_metric,
        "feature_names": list(names),
        "feature_importance": dict(zip(names, map(float, model.feature_importances_))),
    }
    if protocol == "SV":
        shape_metrics = {}
        for shape in sorted({r["asdu_shape"] for r in te}):
            mask = np.asarray([r["asdu_shape"] == shape for r in te]) & known
            if mask.any():
                shape_metrics[str(shape)] = score_metrics(y_test[mask], predictions[mask])
        marker14 = np.asarray([r["asdu_shape"] == "14" and r["marker_positive_audit"] == 1 for r in te]) & known
        details["test_metrics_by_asdu_shape"] = shape_metrics
        details["test_marker_positive_14_asdu_batches"] = {
            "support": int(marker14.sum()),
            "metrics": score_metrics(y_test[marker14], predictions[marker14]) if marker14.any() else {"support": 0},
        }
    evaluation["protocols"][protocol] = details
    models[protocol] = {"model": model, "feature_names": names, "threshold": threshold}
    for row, probability, prediction, label in zip(te, probabilities, predictions, y_test):
        prediction_rows.append({"capture_id": row["capture_id"], "protocol": protocol,
            "stream_id": row["stream_id"], "start_frame": row["start_frame"], "end_frame": row["end_frame"],
            "timestamp_start": row["timestamp_start"], "timestamp_end": row["timestamp_end"],
            "first_attack_timestamp": row["first_attack_timestamp"], "asdu_shape": row["asdu_shape"], "marker_positive_audit": row["marker_positive_audit"],
            "true_attack": "" if label < 0 else int(label), "attack_probability": float(probability),
            "prediction": int(prediction), "label_source": row["label_source"]})
    print(protocol, json.dumps(details, indent=2))

## 5. SV waveform ablation and held-out evaluation

The ablation removes all `wave_*` columns, then repeats capture-grouped OOF threshold selection on training captures only. Both the full model and ablation are evaluated on the same untouched test batches. Test results never select thresholds. SV results are additionally stratified by ASDU count so 1-ASDU versus 14-ASDU packet shape is visible.


In [ ]:
sv_without_wave = [name for name in feature_names("SV") if not name.startswith("wave_")]
sv_train, _, X_sv_train = feature_matrix(train_batches, "SV", sv_without_wave)
sv_test, _, X_sv_test = feature_matrix(test_batches, "SV", sv_without_wave)
y_sv_train = np.asarray([int(r["attack"]) for r in sv_train], dtype=int)
sv_threshold, sv_oof, sv_folds, sv_oof_global, sv_capture_thresholds, sv_capture_fprs = grouped_oof_threshold(
    sv_train, X_sv_train, y_sv_train, CONFIG["threshold"]["max_validation_fpr"])
sv_ablation_model = make_forest()
sv_ablation_model.fit(X_sv_train, y_sv_train, sample_weight=capture_balanced_weights(sv_train))
sv_ablation_prob = sv_ablation_model.predict_proba(X_sv_test)[:, list(sv_ablation_model.classes_).index(1)]
sv_ablation_pred = (sv_ablation_prob >= sv_threshold).astype(int)
sv_full = [r for r in prediction_rows if r["protocol"] == "SV"]
sv_known = np.asarray([str(r["true_attack"]) in {"0", "1"} for r in sv_full])
sv_labels = np.asarray([int(r["true_attack"]) if str(r["true_attack"]) in {"0", "1"} else -1 for r in sv_full])
sv_full_pred = np.asarray([r["prediction"] for r in sv_full])
full_metrics = score_metrics(sv_labels[sv_known], sv_full_pred[sv_known])
abl_metrics = score_metrics(sv_labels[sv_known], sv_ablation_pred[sv_known])
sv_ablation = {"full_threshold": models["SV"]["threshold"], "no_waveform_threshold": sv_threshold,
    "full_features": full_metrics, "without_waveform": abl_metrics,
    "delta_recall_waveform": full_metrics["recall"] - abl_metrics["recall"],
    "delta_false_positive_rate_waveform": full_metrics["false_positive_rate"] - abl_metrics["false_positive_rate"],
    "no_waveform_oof_folds": sv_folds,
    "no_waveform_oof_capture_fprs": sv_capture_fprs,
    "interpretation": "Held-out test comparison; each threshold was selected from training-only grouped OOF predictions. Test results were not used for tuning."}
# Diagnostic only: remove timing and message-shape inputs to see whether the
# counter/waveform signal survives. It does not replace the full model.
sv_counter_wave_names = [name for name in feature_names("SV")
    if not name.startswith(("dt__", "dt_ratio__", "age_since_any_goose_frame__", "no_asdu__"))]
sv_cw_train, _, X_cw_train = feature_matrix(train_batches, "SV", sv_counter_wave_names)
sv_cw_test, _, X_cw_test = feature_matrix(test_batches, "SV", sv_counter_wave_names)
y_cw_train = np.asarray([int(r["attack"]) for r in sv_cw_train], dtype=int)
cw_threshold, cw_oof, cw_folds, cw_oof_global, cw_capture_thresholds, cw_capture_fprs = grouped_oof_threshold(
    sv_cw_train, X_cw_train, y_cw_train, CONFIG["threshold"]["max_validation_fpr"])
cw_model = make_forest()
cw_model.fit(X_cw_train, y_cw_train, sample_weight=capture_balanced_weights(sv_cw_train))
cw_prob = cw_model.predict_proba(X_cw_test)[:, list(cw_model.classes_).index(1)]
cw_pred = (cw_prob >= cw_threshold).astype(int)
cw_metrics = score_metrics(sv_labels[sv_known], cw_pred[sv_known])
evaluation["sv_waveform_ablation"] = sv_ablation
evaluation["sv_without_timing_or_shape_diagnostic"] = {
    "removed_prefixes": ["dt__", "dt_ratio__", "age_since_any_goose_frame__", "no_asdu__"],
    "feature_names": sv_counter_wave_names, "threshold": cw_threshold,
    "threshold_provenance": "capture-grouped OOF training-only threshold at configured maximum FPR",
    "oof_metrics": cw_oof_global, "oof_fpr_by_capture": cw_capture_fprs,
    "test_metrics_known_labels": cw_metrics,
    "test_metrics_by_asdu_shape": {
        shape: score_metrics(sv_labels[np.asarray([r["asdu_shape"] == shape for r in sv_full]) & sv_known],
                             cw_pred[np.asarray([r["asdu_shape"] == shape for r in sv_test]) & sv_known])
        for shape in sorted({r["asdu_shape"] for r in sv_test})
        if np.any(np.asarray([r["asdu_shape"] == shape for r in sv_test]) & sv_known)
    },
}
print(json.dumps({"waveform_ablation": sv_ablation,
                  "without_timing_or_shape": evaluation["sv_without_timing_or_shape_diagnostic"]}, indent=2))
print("Top SV feature importances:", sorted(evaluation["protocols"]["SV"]["feature_importance"].items(), key=lambda x: x[1], reverse=True)[:20])

## 6. Stateful inference, artifact saving, and parity

In [ ]:
import csv
from datetime import datetime, timezone
import math
import time

def _feature_runtime_config(config):
    return {key: config[key] for key in ("batch", "sv_channels", "waveform", "counter", "timing")}

def _score_emitted(rows, bundle):
    """Score only newly emitted batches; memory stays bounded by the caller's chunk."""
    scored = {}
    for protocol in ("GOOSE", "SV"):
        subset = [row for row in rows if row["protocol"] == protocol]
        if not subset:
            continue
        entry = bundle["models"][protocol]
        names = entry["feature_names"]
        X = np.asarray([[float(row[name]) for name in names] for row in subset], dtype=float)
        probabilities = entry["model"].predict_proba(X)[:, list(entry["model"].classes_).index(1)]
        for row, probability in zip(subset, probabilities):
            scored[id(row)] = {**row, "attack_probability": float(probability),
                "prediction": int(probability >= entry["threshold"])}
    return [scored[id(row)] for row in rows]

def classify_event_chunks(chunks, bundle, config):
    """Yield scores incrementally and retain decoder feature history across chunks."""
    if _feature_runtime_config(config) != bundle["feature_config"]:
        raise ValueError("Inference feature config differs from the saved training bundle")
    builder = StatefulBatchFeatureBuilder(config)
    for chunk in chunks:
        emitted = builder.update(chunk)
        if emitted:
            yield from _score_emitted(emitted, bundle)
    final_rows = builder.flush()
    if final_rows:
        yield from _score_emitted(final_rows, bundle)

def classify_pcapng(path, bundle, config):
    from pcap_events import decode_pcapng
    events, diagnostics = decode_pcapng(path)
    if diagnostics["malformed_iec_frames"] or diagnostics["unsupported_linktype"]:
        raise ValueError(f"Capture cannot be fully decoded: {diagnostics}")
    if any(not math.isfinite(float(row["timestamp"])) for row in events):
        raise ValueError("Capture includes events without finite timestamps")
    return list(classify_event_chunks([events], bundle, config)), diagnostics

def classify_ethernet_frames(frame_iter, bundle, config, session_id):
    """Raw-frame adapter: each item is (frame_index, timestamp, packet_bytes)."""
    from pcap_events import decode_ethernet_frame
    def decoded_chunks():
        for frame_index, timestamp, packet in frame_iter:
            events, error = decode_ethernet_frame(packet, timestamp, session_id, frame_index)
            if error:
                raise ValueError(f"Malformed IEC frame {frame_index}: {error}")
            if events:
                yield events
    yield from classify_event_chunks(decoded_chunks(), bundle, config)

# Mixed protocol, multi-ASDU, and capture-boundary parity. Every selected capture
# contributes a contiguous packet prefix so frame order and state reset are checked.
parity_events = []
for source_rows in (train_events, test_events):
    capture_ids = sorted({r["capture_id"] for r in source_rows})
    for capture in capture_ids:
        part = [r for r in source_rows if r["capture_id"] == capture]
        parity_events.extend(part[:700])
assert {r["protocol"] for r in parity_events} == {"GOOSE", "SV"}
assert any(r["protocol"] == "SV" and int(r["no_asdu"]) == 14 for r in parity_events)
# Verify the raw-frame adapter and the PCAPNG reader share byte-for-byte decoding.
from pcap_events import _blocks, decode_ethernet_frame
parity_capture = train_paths[0]
direct_raw = []
for frame_index, (linktype, packet, timestamp) in enumerate(_blocks(parity_capture)):
    if linktype != 1 or frame_index >= 120:
        continue
    decoded, error = decode_ethernet_frame(packet, timestamp, parity_capture.name, frame_index)
    if error:
        raise ValueError(f"Raw frame decoder failed on frame {frame_index}: {error}")
    direct_raw.extend(decoded)
from_pcap = [r for r in train_raw if r["source_pcap"] == parity_capture.name and int(r["frame_index"]) < 120]
assert direct_raw == from_pcap, "Live raw-frame and PCAPNG decode differ"
parity_all = build_batches(parity_events, CONFIG)
parity_chunked = build_batches(parity_events, CONFIG, chunk_size=137)
assert parity_all == parity_chunked, "Feature batches differ across chunk boundaries/capture reset"
# Processing each capture independently must match the same sequence with automatic resets.
independent_batches = []
for capture in dict.fromkeys(r["capture_id"] for r in parity_events):
    independent_batches.extend(build_batches([r for r in parity_events if r["capture_id"] == capture], CONFIG))
assert parity_all == independent_batches, "Feature state leaked across capture boundaries"
provisional_bundle = {"models": models, "config": CONFIG,
    "feature_config": _feature_runtime_config(CONFIG)}
predictions_all = list(classify_event_chunks([parity_events], provisional_bundle, CONFIG))
predictions_chunked = list(classify_event_chunks(
    [parity_events[i:i+137] for i in range(0, len(parity_events), 137)], provisional_bundle, CONFIG))
def prediction_key(row):
    return (row["capture_id"], row["protocol"], row["stream_id"], row["start_frame"], row["end_frame"])
all_by_key = {prediction_key(row): row for row in predictions_all}
chunked_by_key = {prediction_key(row): row for row in predictions_chunked}
assert all_by_key.keys() == chunked_by_key.keys(), "Batch identities differ across chunk boundaries"
for key in all_by_key:
    left, right = all_by_key[key], chunked_by_key[key]
    assert left["prediction"] == right["prediction"], f"Prediction differs for {key}"
    assert np.isclose(left["attack_probability"], right["attack_probability"], rtol=0.0, atol=1e-12), f"Probability differs for {key}"
print("Raw-frame decode parity, mixed-protocol/multi-ASDU feature parity, prediction parity, and capture reset: PASS")

output_dir = Path(CONFIG["paths"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)
source_hashes = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in train_paths + [test_path]}
bundle = {"artifact_type": "iec61850_packet_batch_random_forest",
    "status": "research_only_provisional_lab_labels", "created_utc": datetime.now(timezone.utc).isoformat(),
    "config": CONFIG, "feature_config": _feature_runtime_config(CONFIG), "models": models,
    "feature_pipeline": "tools/batch_pipeline.py", "decoder": "tools/pcap_events.py", "source_sha256": source_hashes}
joblib.dump(bundle, output_dir / "iec61850_rf_batch_bundle.joblib")
evaluation["source_sha256"] = source_hashes
evaluation["config"] = {key: value for key, value in CONFIG.items() if key != "labels"}
(output_dir / "evaluation.json").write_text(json.dumps(evaluation, indent=2, default=str) + "\n", encoding="utf-8")
with (output_dir / "test_batch_predictions.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(prediction_rows[0]))
    writer.writeheader(); writer.writerows(prediction_rows)
# Latency view: batch span and refresh interval per stream; predicted positive
# batches that overlap labeled attacks give an offline frame-to-alert delay estimate.
latency = {}
for protocol in ("GOOSE", "SV"):
    subset = [r for r in prediction_rows if r["protocol"] == protocol]
    by_stream = {}
    for row in subset:
        by_stream.setdefault((row["capture_id"], row["stream_id"]), []).append(row)
    spans, refresh = [], []
    for stream_rows in by_stream.values():
        stream_rows.sort(key=lambda r: r["timestamp_end"])
        spans.extend(float(r["timestamp_end"]) - float(r["timestamp_start"]) for r in stream_rows)
        refresh.extend(float(b["timestamp_end"]) - float(a["timestamp_end"]) for a, b in zip(stream_rows, stream_rows[1:]))
    delays = [float(r["timestamp_end"]) - float(r["first_attack_timestamp"])
        for r in subset if int(r["prediction"]) == 1 and str(r["true_attack"]) == "1" and r["first_attack_timestamp"] != ""]
    latency[protocol] = {
        "median_batch_span_s": float(np.median(spans)) if spans else None,
        "p95_batch_span_s": float(np.quantile(spans, 0.95)) if spans else None,
        "median_score_refresh_s": float(np.median(refresh)) if refresh else None,
        "median_labeled_attack_batch_to_score_s": float(np.median(delays)) if delays else None,
    }
evaluation["offline_latency_summary"] = latency
evaluation["decode_elapsed_s"] = decode_elapsed_s
evaluation["feature_elapsed_s"] = feature_elapsed_s
(output_dir / "evaluation.json").write_text(json.dumps(evaluation, indent=2, default=str) + "\n", encoding="utf-8")
print("Offline latency summary:", json.dumps(latency, indent=2))
print("Saved RF bundle, evaluation, and test batch predictions to", output_dir)

## 7. Interpreting and deploying the result

Each row scored by the forest represents a configurable batch of GOOSE frames or SV Ethernet frames for one logical stream. The SV decoder expands all ASDUs, then the feature builder tracks `smpCnt` and waveform values by ASDU position across frames and collapses each packet to one frame point. Batches aggregate those points; attack labels are positive if any frame/ASDU is labeled attack.

For real-time operation, use `decode_ethernet_frame` for each received Ethernet frame, keep a `StatefulBatchFeatureBuilder` alive across input chunks, and call the corresponding model on each emitted batch. The notebook parity check exercises offline chunks; a production adapter still needs to preserve frame order and flush at session boundaries.

Inspect the held-out metrics, the waveform ablation, class support, and top feature importances together. Test markers provide provisional weak labels. Validate channel interpretation, labeling, publisher topology, false alarms per hour, detection delay, and benign faults against independently verified site captures before operational use. The saved joblib bundle includes models and feature names; deployment must implement the same decoder, batch state, and feature pipeline.